In [6]:
from dotenv import load_dotenv
load_dotenv()


True

In [7]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "/Users/yashgiradkar/Development/projects/LLMops-Project/data/goldens.csv"
df.to_csv(csv_path, index=False)



In [8]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)



{'example_ids': ['e704fe2c-8fdd-4a91-9abf-a54a5c851a37',
  '59901788-bc62-4caf-8f25-525c9c441c09',
  '2a5e9ddb-4ce5-4334-908c-8c24d2feb5fc'],
 'count': 3,
 'as_of': '2026-07-26T11:50:43.317829999Z'}

In [16]:
import sys
sys.path.append("/Users/yashgiradkar/Development/projects/LLMops-Project")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "/Users/yashgiradkar/Development/projects/LLMops-Project/data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 2500,
    chunk_overlap: int = 20,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


Adapter for local file paths to work with ChatIngestor.


    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    


In [17]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])



{"timestamp": "2026-07-26T11:56:04.453006Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T11:56:04.453944Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T11:56:04.454581Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T11:56:04.455179Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T11:56:04.459007Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260726_172604_d3d7c63e", "temp_dir": "data/session_20260726_172604_d3d7c63e", "faiss_dir": "faiss_index/session_20260726_172604_d3d7c63e", "sessionized": true, "timestamp": "2026-07-26T11:56:04.460844Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: Error: Error in [/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py] at line [57] | Message: Invocation error in ConversationalRAG
Traceback:
Traceback (most recent call last):
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib

In [18]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")




{"timestamp": "2026-07-26T11:56:19.084641Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T11:56:19.085239Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T11:56:19.085614Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T11:56:19.086034Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T11:56:19.088650Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260726_172619_75526af6", "temp_dir": "data/session_20260726_172619_75526af6", "faiss_dir": "faiss_index/session_20260726_172619_75526af6", "sessionized": true, "timestamp": "2026-07-26T11:56:19.089620Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as

Testing all questions from the dataset:



{"timestamp": "2026-07-26T11:56:20.023914Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T11:56:20.024793Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T11:56:20.025199Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T11:56:20.025554Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T11:56:20.027945Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-flash", "timestamp": "2026-07-26T11:56:20.028370Z", "level": "info", "event": "Loading LLM"}
{"session_id": "session_20260726_172619_75526af6", "timestamp": "2026-07-26T11:56:20.030871Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260726_172619_75526af6", "timesta

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: Error: Error in [/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py] at line [57] | Message: Invocation error in ConversationalRAG
Traceback:
Traceback (most recent call last):
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14

{"timestamp": "2026-07-26T11:56:23.582480Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T11:56:23.583419Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T11:56:23.583911Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T11:56:23.584317Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T11:56:23.587061Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-flash", "timestamp": "2026-07-26T11:56:23.587513Z", "level": "info", "event": "Loading LLM"}
{"session_id": "session_20260726_172622_475e7a9c", "timestamp": "2026-07-26T11:56:23.590105Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260726_172622_475e7a9c", "timesta

Q2: What percentage of respondents are using RAG in some form?
A2: Error: Error in [/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py] at line [57] | Message: Invocation error in ConversationalRAG
Traceback:
Traceback (most recent call last):
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_inte

{"timestamp": "2026-07-26T11:56:27.296446Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T11:56:27.297412Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T11:56:27.297897Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T11:56:27.298305Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T11:56:27.300894Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-flash", "timestamp": "2026-07-26T11:56:27.301324Z", "level": "info", "event": "Loading LLM"}
{"session_id": "session_20260726_172626_fa105aa7", "timestamp": "2026-07-26T11:56:27.303755Z", "level": "info", "event": "LLM loaded successfully"}
{"session_id": "session_20260726_172626_fa105aa7", "timesta

Q3: How often are most respondents updating their models?
A3: Error: Error in [/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py] at line [57] | Message: Invocation error in ConversationalRAG
Traceback:
Traceback (most recent call last):
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/google/api_core/grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/yashgiradkar/Development/projects/LLMops-Project/.venv/lib/python3.14/site-packages/grpc/_intercept

In [ ]:
from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# ----------------------------
# LLM Judge
# ----------------------------

judge = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

evaluation_prompt = ChatPromptTemplate.from_template(
    """
You are an expert evaluator.

Question:
{question}

Expected Answer:
{reference_answer}

Model Answer:
{prediction}

Determine whether the model answer is factually correct.

Return ONLY one word:

CORRECT

or

INCORRECT
"""
)


# ----------------------------
# Custom Evaluator
# ----------------------------

def correctness_evaluator(inputs, outputs, reference_outputs):

    prompt = evaluation_prompt.format_messages(
        question=inputs["question"],
        prediction=outputs["answer"],
        reference_answer=reference_outputs["answer"],
    )

    response = judge.invoke(prompt)

    result = response.content.strip().upper()

    return {
        "key": "correctness",
        "score": result == "CORRECT",
        "comment": result,
    }


# ----------------------------
# LangSmith Client
# ----------------------------

client = Client()

results = client.evaluate(
    target=answer_ai_report_question,
    data="AgenticAIReportGoldens",
    evaluators=[correctness_evaluator],
    experiment_prefix="test-agenticAIReport-qa-rag",
    metadata={
        "variant": "RAG with FAISS",
        "chunk_size": 2500,
        "chunk_overlap": 20,
        "k": 5,
    },
)

print(results)

# ## Custom Correctness Evaluator
# 
# Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment
# 



    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    


# ### Run Evaluation with Custom Correctness Evaluator
# 



In [ ]:
from langsmith import Client

# Create LangSmith client
client = Client()

# Define evaluators
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = client.evaluate(
    target=answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.0-flash",
        "chunk_size": 2500,
        "chunk_overlap": 20,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

View the evaluation results for experiment: 'agenticAIReport-correctness-eval-6f1594fc' at:
https://smith.langchain.com/o/61be93fe-4ebe-4549-8c06-c19c7fb77195/datasets/00712de2-3a7a-41c5-8d9e-1ba068960b99/compare?selectedSessions=480872e2-152b-4f81-bace-2157fccb76f5




0it [00:00, ?it/s]{"timestamp": "2026-07-26T12:09:21.130702Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-07-26T12:09:21.131145Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-07-26T12:09:21.131524Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_ya...", "GOOGLE_API_KEY": "AQ.Ab8..."}, "timestamp": "2026-07-26T12:09:21.131978Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-07-26T12:09:21.133979Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260726_173921_ed530be4", "temp_dir": "data/session_20260726_173921_ed530be4", "faiss_dir": "faiss_index/session_20260726_173921_ed530be4", "sessionized": true, "timestamp": "2026-07-26T12:09:21.134732Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Repo


Evaluation completed! Check the LangSmith UI for detailed results.


# ### Optional: Combine Multiple Evaluators
# 
# You can use multiple evaluators together to get different perspectives on your RAG system's performance.
# 



Run evaluation with multiple evaluators
Uncomment to run:
experiment_results_combined = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=combined_evaluators,
    experiment_prefix="agenticAIReport-multi-eval",
    description="Evaluating RAG system with multiple evaluators",
    metadata={
        "variant": "RAG with FAISS",
        "evaluators": "correctness + cot_qa",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)


# 

